In [ ]:
import pandas as pd

df = pd.read_csv(r"/home/ayoub/Desktop/darkom_annonces/data/darkom_annonces.csv")

df.info()

In [ ]:
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

df.info()

In [ ]:
df.head()

In [ ]:
df["date_publication"] = pd.to_datetime(df["date_publication"], format="%Y-%m-%d", errors="coerce")

df[["type_bien", "transaction"]] = df[
    ["type_bien", "transaction"]
].astype("category")

df["annee_construction"] = df["annee_construction"].astype("Int64")

In [ ]:
numeric_columns = [
    "prix",
    "nb_chambres",
    "nb_salles_bain",
    "etage",
    "annee_construction"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).round().astype("Int64")

df.info()

In [ ]:
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

df = df.sort_values("date_publication").reset_index(drop=True)

df["date_publication"] = (
    df["date_publication"]
    .ffill()
    .bfill()
)

In [ ]:
df["quartier"] = df.groupby("ville")["quartier"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

In [ ]:
df.info()

In [ ]:
df["annee_construction"].unique()

In [ ]:
df["type_bien"].unique()

In [ ]:
def extract_type_bien(title):

    title = title.lower()

    if "appartement" in title:
        return "Appartement"

    elif "villa" in title:
        return "Villa"

    elif "bureau" in title:
        return "Bureau"

    elif "terrain" in title:
        return "Terrain"

    elif "duplex" in title:
        return "Duplex"

    else:
        return None

df["type_bien"] = df["type_bien"].fillna(
    df["titre"].apply(extract_type_bien)
)


In [ ]:
df["annee_construction"] = df.groupby(
    ["ville", "type_bien"]
)["annee_construction"].transform(
    lambda x: x.fillna(round(x.median()))
)

df.info()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.groupby("transaction")["prix"].describe()

In [ ]:
df.groupby("transaction")["prix"].quantile([0.90, 0.95, 0.99])

In [ ]:
# Function to calculate outlier percentage using IQR

def outlier_percentage(df, column):

    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_count = len(outliers)

    total_count = len(df)

    percentage = (
        outlier_count / total_count
    ) * 100

    print(f"\nColumn: {column}")

    print(f"Outliers: {outlier_count}")

    print(f"Percentage: {percentage:.2f}%")

In [ ]:
outlier_percentage(df, "prix")

outlier_percentage(df, "surface")

outlier_percentage(df, "nb_chambres")

outlier_percentage(df, "nb_salles_bain")

In [ ]:
df = df[df["nb_chambres"] <= 10]

df = df[df["nb_salles_bain"] <= 6]

df = df[df["prix"] <= 20000000]

In [ ]:
df.info()

In [ ]:
df.info()

In [ ]:
# Replace unrealistic nb_chambres values

mask = df["nb_chambres"] > 10

df.loc[mask, "nb_chambres"] = df.groupby(
    ["ville", "type_bien"]
)["nb_chambres"].transform("median")

In [ ]:
# Replace unrealistic nb_salles_bain values

mask = df["nb_salles_bain"] > 6

df.loc[mask, "nb_salles_bain"] = df.groupby(
    ["ville", "type_bien"]
)["nb_salles_bain"].transform("median")

In [ ]:
mask = df["prix"] > 20000000

median_prices = df.groupby(
    ["ville", "type_bien"]
)["prix"].transform("median").round()

df.loc[mask, "prix"] = median_prices[mask]

df["prix"] = df["prix"].astype("Int64")

In [ ]:
df.info()

In [ ]:
df.head(30)

In [ ]:
df.info()

In [ ]:
dl = pd.read_csv(r"/home/ayoub/Desktop/darkom_annonces/data/cleaned_data.csv")

dl.info()

In [150]:
import pandas as pd
df = pd.read_csv(r"/home/ayoub/Desktop/darkom_annonces/data/cleaned_feature_eng_data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   annonce_id             1447 non-null   str    
 1   date_publication       1447 non-null   str    
 2   titre                  1447 non-null   str    
 3   ville                  1447 non-null   str    
 4   quartier               1447 non-null   str    
 5   type_bien              1447 non-null   str    
 6   transaction            1447 non-null   str    
 7   prix                   1447 non-null   int64  
 8   surface                1447 non-null   float64
 9   nb_chambres            1447 non-null   int64  
 10  nb_salles_bain         1447 non-null   int64  
 11  etage                  1447 non-null   int64  
 12  annee_construction     1447 non-null   int64  
 13  prix_m2                1447 non-null   int64  
 14  age_bien               1447 non-null   int64  
 15  categorie_prix 

In [151]:
df.head()

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction,prix_m2,age_bien,categorie_prix,categorie_surface,annee_publication,mois_publication,trimestre_publication
0,ANO000708,2023-01-01,Villa moderne Agadir,Agadir,Talborjt,Villa,Vente,2592302,248.46,6,4,0,2007,10433,19,Haut Standing,Grand,2023,1,1
1,ANO001039,2023-01-01,Appartement à vente - Kenitra,Kenitra,Hay Essalam,Appartement,Vente,2155591,91.29,2,1,2,2008,23613,18,Haut Standing,Moyen,2023,1,1
2,ANO000870,2023-01-02,Bureau à vente - Tétouan,Tetouan,M'diq,Bureau,Vente,317774,155.88,0,0,1,2020,2039,6,Moyen,Grand,2023,1,1
3,ANO000036,2023-01-03,Appartement 132m² Meknès,Meknes,Hamria,Appartement,Vente,598486,132.15,1,1,2,2011,4529,15,Moyen,Moyen,2023,1,1
4,ANO001275,2023-01-03,Beau Villa Meknès,Meknes,Hamria,Villa,Vente,2177553,264.60,4,2,0,2004,8230,22,Haut Standing,Grand,2023,1,1


In [149]:
df["surface"].value_counts().sort_values()

surface
61.95      1
285.36     1
97.86      1
116.79     1
49.62      1
          ..
8.00       4
3000.00    5
12.00      6
15.00      8
2500.00    9
Name: count, Length: 1398, dtype: int64

In [133]:
# Create price per square meter

df["prix_m2"] = (
    df["prix"] /
    df["surface"]
).round().astype("Int64")

In [134]:
df["prix_m2"].head()

0    10433
1    23613
2     2039
3     4529
4      230
Name: prix_m2, dtype: Int64

In [136]:
df.head()

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction,prix_m2,age_bien
0,ANO000708,2023-01-01,Villa moderne Agadir,Agadir,Talborjt,Villa,Vente,2592302,248.46,6,4,0,2007,10433,19
1,ANO001039,2023-01-01,Appartement à vente - Kenitra,Kenitra,Hay Essalam,Appartement,Vente,2155591,91.29,2,1,2,2008,23613,18
2,ANO000870,2023-01-02,Bureau à vente - Tétouan,Tetouan,M'diq,Bureau,Vente,317774,155.88,0,0,1,2020,2039,6
3,ANO000036,2023-01-03,Appartement 132m² Meknès,Meknes,Hamria,Appartement,Vente,598486,132.15,1,1,2,2011,4529,15
4,ANO000883,2023-01-03,Terrain à location - Meknès,Meknes,Ville Nouvelle,Terrain,Location,2755,12.00,0,0,0,2009,230,17
